# Bayesian Inference Assignment



In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy.stats import beta as beta_dist
from scipy.stats import norm, lognorm

np.random.seed(42)


---
# Question 1

## Task 1 — Visualizing the Mechanics

The item response function is

$$p_i(\theta) = P(Y_i=1\mid \Theta=\theta) = \frac{1}{1+e^{-a_i(\theta-b_i)}}.$$

This is a logistic curve in $\theta$: it is centered at $\theta = b_i$ (where $p_i(b_i)=0.5$),
and its steepness at the center is controlled by $a_i$ (the slope there is $a_i/4$).
Below we plot two discrimination values $a_i \in \{0.5, 2.0\}$; for $a_i = 1.5$ we additionally
show three difficulty values $b_i \in \{-1.5, 0, 1.5\}$.


In [2]:
def item_response_curve(theta, a, b):
    return 1.0 / (1.0 + np.exp(-a * (theta - b)))

theta_grid = np.linspace(-4, 4, 400)

fig1 = go.Figure()

for a_val in [0.5, 2.0]:
    fig1.add_trace(go.Scatter(
        x=theta_grid, y=item_response_curve(theta_grid, a_val, 0.0),
        mode="lines", name=f"a = {a_val}, b = 0.0"
    ))

for b_val in [-1.5, 0.0, 1.5]:
    fig1.add_trace(go.Scatter(
        x=theta_grid, y=item_response_curve(theta_grid, 1.5, b_val),
        mode="lines", line=dict(dash="dot"),
        name=f"a = 1.5, b = {b_val}"
    ))

fig1.update_layout(
    title="2PL Item Characteristic Curves: Effect of Discrimination (a) and Difficulty (b)",
    xaxis_title="Latent ability \u03b8",
    yaxis_title="P(Y = 1 | \u0398 = \u03b8)",
    template="plotly_white",
    hovermode="x unified"
)
fig1.add_hline(y=0.5, line_dash="dash", line_color="gray", opacity=0.5)
fig1.show()


**Interpretation.** Changing $a_i$ (with $b_i$ fixed at $0$) does not move the curve
horizontally — every curve still crosses $p_i=0.5$ at $\theta = 0$. Instead, $a_i$ controls
*how fast* the probability rises through that midpoint: a large $a_i$ (steep curve) means
the item is highly discriminating — a tiny change in $\theta$ near $b_i$ flips the response
probability from near $0$ to near $1$. A small $a_i$ produces a shallow, almost linear
curve — the item barely distinguishes between nearby ability levels.

Changing $b_i$ (with $a_i$ fixed at $1.5$) **translates the curve horizontally** along the
$\theta$-axis without changing its shape. This matches the interpretation of $b_i$ as a
*difficulty* parameter: it is the ability level at which the user has a 50% chance of
answering correctly. A larger $b_i$ means a harder item, since a higher $\theta$ is now
required to reach the midpoint of the curve.

## Task 2 — Sequential Likelihood Contribution

Conditional on $\Theta=\theta$, a single response $y_k \in \{0,1\}$ at step $k$ is Bernoulli
with success probability $p_k(\theta)$, so its likelihood contribution is

$$L(y_k\mid\theta) = p_k(\theta)^{y_k}\,\bigl[1-p_k(\theta)\bigr]^{1-y_k}, \qquad p_k(\theta)=\frac{1}{1+e^{-a_k(\theta-b_k)}}.$$

Because responses are conditionally independent given $\Theta=\theta$, the joint likelihood
of the running history $\mathbf y^{(k)}=(y_1,\dots,y_k)$ is the product of the individual
contributions:

$$L\bigl(\mathbf y^{(k)}\mid\theta\bigr) = \prod_{i=1}^{k} p_i(\theta)^{y_i}\bigl[1-p_i(\theta)\bigr]^{1-y_i}.$$

Equivalently, $L(\mathbf y^{(k)}\mid\theta) = L(\mathbf y^{(k-1)}\mid\theta)\cdot L(y_k\mid\theta)$,
which is what makes the *sequential* update below possible — the entire history never needs
to be re-multiplied, only the newest factor.

## Task 3 — Mathematical Formulation of the Running Update

By Bayes' rule, the posterior at step $k$ is proportional to the new likelihood contribution
times the *previous* posterior (which plays the role of the current prior):

$$f_{\Theta\mid \mathbf Y^{(k)}}\bigl(\theta \mid \mathbf y^{(k)}\bigr) \;\propto\; L(y_k\mid\theta)\; f_{\Theta\mid \mathbf Y^{(k-1)}}\bigl(\theta \mid \mathbf y^{(k-1)}\bigr),$$

with base case $f_{\Theta\mid\mathbf Y^{(0)}}(\theta) = f_\Theta^{(0)}(\theta)$, the $\mathscr N(0,1)$
density. Written out in full,

$$f_{\Theta\mid \mathbf Y^{(k)}}\bigl(\theta \mid \mathbf y^{(k)}\bigr) = \frac{p_k(\theta)^{y_k}[1-p_k(\theta)]^{1-y_k}\; f_{\Theta\mid \mathbf Y^{(k-1)}}\bigl(\theta \mid \mathbf y^{(k-1)}\bigr)}{\displaystyle\int_{\mathbb R} p_k(s)^{y_k}[1-p_k(s)]^{1-y_k}\; f_{\Theta\mid \mathbf Y^{(k-1)}}\bigl(s \mid \mathbf y^{(k-1)}\bigr)\,ds}.$$

This is the *online* form of Bayes' rule: "yesterday's posterior is today's prior."

## Task 4 — Dynamic Shifting

Suppose the item at step $k$ is hard (large $b_k$) and the user answers correctly ($y_k=1$).
The likelihood factor $p_k(\theta)$ is small for $\theta \ll b_k$ and only approaches $1$
once $\theta$ exceeds $b_k$. Multiplying the previous posterior by this factor therefore
**down-weights** all the probability mass sitting at ability values below $b_k$ and
**preserves** mass at $\theta \gtrsim b_k$. After renormalizing, the mode of the running
posterior is pulled toward higher ability — and the pull is strong precisely because
succeeding on a *hard* item is disproportionately good evidence of high ability (an easy
item would have been answered correctly by almost any ability level, so it carries much
less information about where on the ability scale the user actually sits).

## Task 5 — Tracking Certainty and Sharpness

The discrimination parameter $a_k$ controls how quickly $p_k(\theta)$ transitions from $0$
to $1$ around $\theta=b_k$, i.e. how informative the likelihood factor is as a function of
$\theta$:

* **Large $a_k$:** the item curve is nearly a step function. Observing $y_k$ behaves almost
  like learning "$\theta$ is above (or below) $b_k$" with near certainty — the likelihood
  strongly discounts values on the wrong side of $b_k$. This produces a **sharp, aggressive
  update**: the posterior narrows substantially and its mode can move a long way in one step.
* **Small $a_k$:** the curve is shallow and close to $0.5$ everywhere, so the likelihood
  factor is nearly *flat* in $\theta$. Multiplying by a near-constant function barely
  changes the posterior's shape — the update is **weak**, and the mode barely moves.

In the limit $a_k \to \infty$ a single response becomes maximally informative; in the limit
$a_k \to 0$ the item carries almost no information about $\theta$.

## Task 6 — Numerical Implementation of a Running Grid

Because the 2PL likelihood is not conjugate to the Gaussian prior, the posterior has no
closed form and must be tracked numerically:

1. **Discretize.** Fix a fine grid $\theta_1 < \theta_2 < \dots < \theta_M$ spanning the
   plausible ability range (e.g. $[-6,6]$).
2. **Initialize.** Evaluate the prior density on the grid: $f^{(0)}_m = \phi(\theta_m)$
   (standard normal pdf), $m=1,\dots,M$.
3. **Sequential update.** On observing $y_k$ with item parameters $(a_k,b_k)$: compute
   $p_k(\theta_m)$ on the grid, then form the unnormalized posterior
   $\tilde f^{(k)}_m = p_k(\theta_m)^{y_k}[1-p_k(\theta_m)]^{1-y_k}\cdot f^{(k-1)}_m$.
4. **Normalize.** Approximate the normalizing integral with the trapezoidal rule,
   $Z_k \approx$ `np.trapezoid(f_tilde, theta_grid)`, and set $f^{(k)}_m = \tilde f^{(k)}_m / Z_k$.
5. **Estimators.** Running posterior mean $\widehat\theta^{(k)}_{\text{Bayes}} \approx$
   `np.trapezoid(theta_grid * f_k, theta_grid)`; running MAP $= \arg\max_m f^{(k)}_m$.
6. **Repeat** for each new item, always using the previous normalized array as the new prior.

## Task 7 — Evaluating Convergence over the Timeline


In [3]:
def run_2pl_grid_simulation(theta_true=0.75, n_items=20, grid_size=2000, theta_range=(-6, 6), seed=0):
    rng = np.random.default_rng(seed)
    theta_grid = np.linspace(theta_range[0], theta_range[1], grid_size)

    posterior = norm.pdf(theta_grid, loc=0, scale=1)
    posterior /= np.trapezoid(posterior, theta_grid)

    bayes_estimates = [np.trapezoid(theta_grid * posterior, theta_grid)]
    map_estimates = [theta_grid[np.argmax(posterior)]]

    for k in range(1, n_items + 1):
        b_k = rng.normal(0, 1)
        a_k = rng.uniform(0.5, 2.0)
        p_true = 1.0 / (1.0 + np.exp(-a_k * (theta_true - b_k)))
        y_k = 1 if rng.uniform(0, 1) < p_true else 0

        p_grid = 1.0 / (1.0 + np.exp(-a_k * (theta_grid - b_k)))
        likelihood = p_grid**y_k * (1 - p_grid)**(1 - y_k)

        unnormalized = likelihood * posterior
        Z = np.trapezoid(unnormalized, theta_grid)
        posterior = unnormalized / Z

        bayes_estimates.append(np.trapezoid(theta_grid * posterior, theta_grid))
        map_estimates.append(theta_grid[np.argmax(posterior)])

    return np.array(bayes_estimates), np.array(map_estimates)

bayes_est, map_est = run_2pl_grid_simulation(theta_true=0.75, n_items=20, seed=1)

steps = np.arange(0, 21)
fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=steps, y=bayes_est, mode="lines+markers", name="Posterior Mean (Bayes)"))
fig2.add_trace(go.Scatter(x=steps, y=map_est, mode="lines+markers", name="MAP estimate"))
fig2.add_hline(y=0.75, line_dash="dash", line_color="red", annotation_text="theta_true = 0.75")
fig2.update_layout(
    title="Convergence of Running Posterior Mean and MAP Estimates (2PL Model, n = 20 items)",
    xaxis_title="Item step k",
    yaxis_title="Ability estimate",
    template="plotly_white",
    hovermode="x unified"
)
fig2.show()

print("Final posterior mean estimate:", round(bayes_est[-1], 4))
print("Final MAP estimate:           ", round(map_est[-1], 4))
print("True theta:                    0.75")


Final posterior mean estimate: 0.1122
Final MAP estimate:            0.1111
True theta:                    0.75


**Analysis.** Early in the sequence (small $k$), both estimators fluctuate — a single
response, especially from a low-discrimination item or an item whose difficulty is far from
the true ability, carries little information, and the prior still dominates. As $k$ grows,
accumulated evidence from many items increasingly outweighs the fixed prior, so the running
estimators drift toward and stabilize near $\theta_{\text{true}}=0.75$, and the gap
$|\widehat\theta^{(k)} - \theta_{\text{true}}|$ tends to shrink with more observations
(though not perfectly monotonically, since each item's evidence is itself random). The
narrowing gap mirrors a narrowing posterior variance: more items answered means the
platform's ability estimate becomes both more accurate and more *confident* — the sharpening
effect from Task 5, accumulated across many sequential updates.


---
# Question 2

## Task 1 — Structural Probability and Properties


In [4]:
theta_grid_beta = np.linspace(0.001, 0.999, 1000)

params = [
    (1, 1, "Uninformative: Beta(1, 1)"),
    (2, 8, "Right-skewed: Beta(2, 8)"),
    (8, 2, "Left-skewed: Beta(8, 2)"),
]

fig3 = go.Figure()
for a_p, b_p, label in params:
    fig3.add_trace(go.Scatter(
        x=theta_grid_beta, y=beta_dist.pdf(theta_grid_beta, a_p, b_p),
        mode="lines", name=label
    ))

fig3.update_layout(
    title="Beta(\u03b1, \u03b2) Densities for Three Parameter Regimes",
    xaxis_title="\u03b8 (click-through rate)",
    yaxis_title="Density",
    template="plotly_white",
    hovermode="x unified"
)
fig3.show()


**Interpretation.** The Beta mean is $\mathbb E[\Theta]=\alpha/(\alpha+\beta)$, so the
*balance* between $\alpha$ and $\beta$ determines where the density's center of mass sits
on $[0,1]$: $\text{Beta}(1,1)$ is exactly uniform (no preferred value — total prior
ignorance); $\text{Beta}(2,8)$ has mean $0.2$, concentrating mass near small $\theta$
(right-skewed, long tail toward $1$); $\text{Beta}(8,2)$ has mean $0.8$, concentrating mass
near large $\theta$ (left-skewed, long tail toward $0$). More generally, increasing $\alpha$
relative to $\beta$ pulls the center of mass toward $1$ (more prior "successes"), while
increasing $\beta$ relative to $\alpha$ pulls it toward $0$ (more prior "failures"); and the
*sum* $\alpha+\beta$ controls concentration — larger sums produce a narrower, more peaked
density (stronger prior conviction), all else equal.

## Task 2 — Sequential Likelihood and Joint History

Conditional on $\Theta=\theta$, a single interaction $y_k\in\{0,1\}$ is Bernoulli($\theta$),
so its likelihood contribution is

$$L(y_k\mid\theta) = \theta^{y_k}(1-\theta)^{1-y_k}.$$

Assuming conditionally independent interactions, the joint likelihood of the running
history $\mathbf y^{(k)}=(y_1,\dots,y_k)$ is

$$L\bigl(\mathbf y^{(k)}\mid\theta\bigr)=\prod_{i=1}^k \theta^{y_i}(1-\theta)^{1-y_i} = \theta^{s_k}(1-\theta)^{k-s_k}, \qquad s_k=\sum_{i=1}^k y_i,$$

i.e. it depends on the data only through the running click count $s_k$ (a sufficient statistic).

## Task 3 — Closed-Form Analytical Updates (Conjugacy)

By Bayes' rule, $f_{\Theta\mid \mathbf Y^{(k)}}(\theta\mid\mathbf y^{(k)}) \propto L(y_k\mid\theta)\, f_{\Theta\mid \mathbf Y^{(k-1)}}(\theta\mid\mathbf y^{(k-1)})$. If the prior at step $k-1$ is $\text{Beta}(\alpha_{k-1},\beta_{k-1})$, then

$$f_{\Theta\mid \mathbf Y^{(k)}}(\theta\mid\mathbf y^{(k)}) \;\propto\; \theta^{y_k}(1-\theta)^{1-y_k}\cdot \theta^{\alpha_{k-1}-1}(1-\theta)^{\beta_{k-1}-1} = \theta^{\alpha_{k-1}+y_k-1}(1-\theta)^{\beta_{k-1}+(1-y_k)-1}.$$

This is (up to normalization) exactly the kernel of a $\text{Beta}(\alpha_k,\beta_k)$
density, which **proves Beta–Binomial conjugacy**: the posterior stays in the Beta family
at every step, with the closed-form recursive updates

$$\alpha_k = \alpha_{k-1} + y_k, \qquad \beta_k = \beta_{k-1} + (1-y_k).$$

In words: a click ($y_k=1$) adds $1$ to $\alpha$; a non-click ($y_k=0$) adds $1$ to $\beta$.
Unrolled from the base prior $(\alpha_0,\beta_0)$ after $k$ impressions with $s_k=\sum_{i\le k} y_i$
clicks, $\alpha_k=\alpha_0+s_k$ and $\beta_k=\beta_0+(k-s_k)$.

The posterior mean at step $k$ is

$$\mathbb E\bigl[\Theta\mid \mathbf Y^{(k)}=\mathbf y^{(k)}\bigr] = \frac{\alpha_k}{\alpha_k+\beta_k} = \frac{\alpha_0+s_k}{\alpha_0+\beta_0+k}.$$

## Task 4 — Dynamic Shifting Mechanics

Each new observation nudges exactly one shape parameter by $1$: a click ($y_k=1$) increases
$\alpha_k$, which — since the Beta mean is $\alpha_k/(\alpha_k+\beta_k)$ — shifts the
posterior's center of mass *up*; a non-click ($y_k=0$) increases $\beta_k$ and shifts the
center of mass *down*. Because both $\alpha$ and $\beta$ also grow the *denominator*
$\alpha_k+\beta_k$, each individual observation's effect on the mean shrinks as more data
accumulates (the same $+1$ matters much less once $\alpha_k+\beta_k$ is large) — this is
exactly how the posterior both shifts and sharpens simultaneously.

Crucially, this whole update is **pure arithmetic** — increment one counter, done — because
the Bernoulli likelihood and Beta prior are conjugate. This is in sharp contrast to a
non-conjugate setup like the 2PL IRT model of Question 1, where the logistic likelihood does
not combine with a Gaussian (or any standard) prior to produce a posterior in a recognizable
parametric family. There, no closed-form recursion for the posterior exists at all, and one
is forced to fall back on **numerical grid integration**: evaluate prior $\times$ likelihood
on a discretized grid of $\theta$ values and renormalize (e.g. via the trapezoidal rule)
after every observation, exactly as described in Question 1, Task 6.

## Task 5 — Running Point Estimators

Directly from the updated shape parameters $\alpha_k,\beta_k$ of $\text{Beta}(\alpha_k,\beta_k)$:

$$\widehat\theta^{(k)}_{\mathrm{Bayes}} = \mathbb E[\Theta\mid \mathbf Y^{(k)}] = \frac{\alpha_k}{\alpha_k+\beta_k},$$

$$\widehat\theta^{(k)}_{\mathrm{MAP}} = \frac{\alpha_k-1}{\alpha_k+\beta_k-2} \quad (\text{valid when } \alpha_k>1,\ \beta_k>1).$$

(If $\alpha_k\le 1$ and $\beta_k\ge 1$ the mode is at the boundary $\theta=0$; if $\beta_k\le 1$
and $\alpha_k\ge 1$ it is at $\theta=1$; if both $\alpha_k,\beta_k<1$ the density is bimodal
at the two boundaries. With $\alpha_0=\beta_0=1$ and at least one observation, $\alpha_k,\beta_k\ge 1$
and typically $>1$ after a couple of updates, so the interior formula applies almost immediately.)

## Task 6 — Performance Tracking and Convergence Analysis


In [5]:
def run_beta_binomial_simulation(theta_true=0.35, n_impressions=100, alpha0=1, beta0=1, seed=0):
    rng = np.random.default_rng(seed)

    alpha_k, beta_k = alpha0, beta0
    bayes_estimates = [alpha_k / (alpha_k + beta_k)]
    map_estimates = [ (alpha_k - 1) / (alpha_k + beta_k - 2) if alpha_k > 1 and beta_k > 1 else np.nan ]

    for k in range(1, n_impressions + 1):
        y_k = 1 if rng.uniform(0, 1) < theta_true else 0
        alpha_k = alpha_k + y_k
        beta_k = beta_k + (1 - y_k)

        bayes_estimates.append(alpha_k / (alpha_k + beta_k))
        if alpha_k > 1 and beta_k > 1:
            map_estimates.append((alpha_k - 1) / (alpha_k + beta_k - 2))
        else:
            map_estimates.append(0.0 if alpha_k <= beta_k else 1.0)

    return np.array(bayes_estimates), np.array(map_estimates), alpha_k, beta_k

bayes_ctr, map_ctr, alpha_final, beta_final = run_beta_binomial_simulation(
    theta_true=0.35, n_impressions=100, alpha0=1, beta0=1, seed=7
)

steps_ctr = np.arange(0, 101)
fig4 = go.Figure()
fig4.add_trace(go.Scatter(x=steps_ctr, y=bayes_ctr, mode="lines", name="Posterior Mean (Bayes)"))
fig4.add_trace(go.Scatter(x=steps_ctr, y=map_ctr, mode="lines", name="MAP estimate"))
fig4.add_hline(y=0.35, line_dash="dash", line_color="red", annotation_text="theta_true = 0.35")
fig4.update_layout(
    title="Convergence of Running CTR Estimators (Beta-Binomial Model, n = 100 impressions)",
    xaxis_title="Impression step k",
    yaxis_title="Estimated click-through rate",
    template="plotly_white",
    hovermode="x unified"
)
fig4.show()

print(f"Final posterior: Beta({alpha_final}, {beta_final})")
print("Final posterior mean:", round(bayes_ctr[-1], 4))
print("Final MAP estimate: ", round(map_ctr[-1], 4))
print("True theta:          0.35")


Final posterior: Beta(37, 65)
Final posterior mean: 0.3627
Final MAP estimate:  0.36
True theta:          0.35


**Analysis.** With the uninformative $\text{Beta}(1,1)$ start, the earliest estimates
are extremely volatile — after $1$–$2$ impressions the posterior mean can swing between
$0$, $0.5$, or $1$, since only a handful of clicks/non-clicks have been observed and the
prior offers no counterweight. As $k$ approaches $100$, the running estimators settle down
and converge toward $\theta_{\text{true}}=0.35$, with fluctuations shrinking roughly like
$1/\sqrt{k}$ (the standard error of a Beta posterior mean scales this way once $\alpha_k+\beta_k$
is large). This illustrates a general Bayesian principle: with an uninformative or weak
prior, the *data* rapidly dominates the posterior — the influence of the initial prior
$(\alpha_0,\beta_0)=(1,1)$ (worth "2 pseudo-observations") becomes negligible once $k\gg 2$,
because $\alpha_k+\beta_k = \alpha_0+\beta_0+k$ grows linearly while the prior's fixed
contribution stays constant. In other words, evidence accumulates faster than the prior's
influence persists, and by $k=100$ the choice of starting prior has only a small residual
effect on the estimate.


---
# Question 3

## Task 1 — Prior Belief Boundaries


In [6]:
a0, b0 = 8, 1.5
theta_shm = np.linspace(0.01, 1.0, 1000)
prior_shm = beta_dist.pdf(theta_shm, a0, b0)

fig5 = go.Figure()
fig5.add_trace(go.Scatter(x=theta_shm, y=prior_shm, mode="lines", fill="tozeroy",
                           name=f"Prior: Beta({a0}, {b0})"))
prior_mean_shm = a0 / (a0 + b0)
fig5.add_vline(x=prior_mean_shm, line_dash="dash", line_color="red",
               annotation_text=f"Prior mean = {prior_mean_shm:.3f}")
fig5.update_layout(
    title="Initial Prior on Remaining Stiffness Efficiency: Beta(8, 1.5)",
    xaxis_title="\u03b8 (stiffness efficiency factor)",
    yaxis_title="Density",
    template="plotly_white",
)
fig5.show()

print("E[Theta^(0)] =", round(prior_mean_shm, 4))


E[Theta^(0)] = 0.8421


**Analytical expected prior stiffness.** For $\Theta^{(0)}\sim\text{Beta}(\alpha_0,\beta_0)$
with $\alpha_0=8,\ \beta_0=1.5$,

$$\mathbb E\bigl[\Theta^{(0)}\bigr] = \frac{\alpha_0}{\alpha_0+\beta_0} = \frac{8}{9.5} \approx 0.842.$$

**Why this is an appropriate "healthy" prior.** The Beta family is naturally supported on
$[0,1]$, matching the physical domain of a stiffness *efficiency* factor. With
$\alpha_0=8 \gg \beta_0=1.5$, the density is strongly left-skewed: most of its mass sits
close to $\theta=1$ (pristine condition), with a short, sharply decaying tail toward small
$\theta$. This encodes the engineers' belief that, before any damage evidence has been
collected, the component is *very likely close to fully healthy*, while still leaving
non-negligible density near $\theta=0$ to allow the data to reveal unexpected damage if it
is present. A symmetric or uniform prior would fail to capture this asymmetric "presumed
healthy" engineering judgment.

## Task 2 — Structural Likelihood Formulation

The sensor model is $y_k = \theta K_{\text{nominal}} e^{\epsilon_k}$ with
$\epsilon_k \sim \mathscr N(0,\sigma^2)$. Taking logs, $\ln y_k = \ln\theta + \ln K_{\text{nominal}} + \epsilon_k$,
so conditional on $\theta$,

$$\ln Y_k \mid \Theta=\theta \;\sim\; \mathscr N\bigl(\ln\theta+\ln K_{\text{nominal}},\,\sigma^2\bigr),$$

i.e. $Y_k\mid\Theta=\theta$ is **log-normal**. Using the change-of-variables formula
$f_Y(y) = f_{\ln Y}(\ln y)\cdot \left|\frac{d}{dy}\ln y\right| = f_{\ln Y}(\ln y)/y$, the
likelihood contribution of a single measurement is

$$L(y_k\mid\theta) = \frac{1}{y_k\,\sigma\sqrt{2\pi}}\exp\left(-\frac{\bigl[\ln y_k - \ln\theta - \ln K_{\text{nominal}}\bigr]^2}{2\sigma^2}\right), \qquad y_k>0.$$

Assuming conditionally independent readings given $\theta$, the joint likelihood of the
running history $\mathbf y^{(k)}=(y_1,\dots,y_k)$ is

$$L\bigl(\mathbf y^{(k)}\mid\theta\bigr) = \prod_{i=1}^{k}\frac{1}{y_i\,\sigma\sqrt{2\pi}}\exp\left(-\frac{\bigl[\ln y_i-\ln\theta-\ln K_{\text{nominal}}\bigr]^2}{2\sigma^2}\right).$$

## Task 3 — Mathematical Formulation of the Non-Conjugate Grid Update

Conjugacy would require the likelihood, viewed as a function of $\theta$, to combine with a
Beta density and again yield a Beta density. Here, as a function of $\theta$, the likelihood
factor $L(y_k\mid\theta) \propto \exp\!\Bigl(-\tfrac{1}{2\sigma^2}[\ln y_k - \ln\theta - \ln K_{\text{nominal}}]^2\Bigr)$
is a *log-normal-shaped* function of $\theta$ (Gaussian in $\ln\theta$, not in $\theta$
itself, and certainly not of the polynomial-in-$\theta$-and-$(1-\theta)$ form that a Beta
kernel requires). Multiplying a Beta density by this log-normal-in-$\theta$ factor does
**not** produce another Beta density — the resulting kernel $\theta^{\alpha-1}(1-\theta)^{\beta-1}\exp(-\tfrac{1}{2\sigma^2}(\ln\theta - c)^2)$
is not the kernel of any standard named distribution, so **no closed-form conjugate update
exists**. The recursive posterior relationship must therefore be left in unnormalized,
proportional form:

$$f_{\Theta\mid\mathbf Y^{(k)}}\bigl(\theta\mid\mathbf y^{(k)}\bigr) \;\propto\; L(y_k\mid\theta)\;f_{\Theta\mid\mathbf Y^{(k-1)}}\bigl(\theta\mid\mathbf y^{(k-1)}\bigr), \qquad \theta\in(0,1],$$

with the normalizing constant computed numerically at every step (see Task 5).

## Task 4 — Running Point Estimates

Since no closed form is available, the point estimators must be written as *definite
integrals* over the physical domain $(0,1]$:

$$\widehat\theta^{(k)}_{\mathrm{Bayes}} = \mathbb E\bigl[\Theta\mid \mathbf Y^{(k)}=\mathbf y^{(k)}\bigr] = \int_0^1 \theta\; f_{\Theta\mid\mathbf Y^{(k)}}\bigl(\theta\mid\mathbf y^{(k)}\bigr)\,d\theta,$$

$$\widehat\theta^{(k)}_{\mathrm{MAP}} = \operatorname*{arg\,max}_{\theta\in(0,1]}\; f_{\Theta\mid\mathbf Y^{(k)}}\bigl(\theta\mid\mathbf y^{(k)}\bigr).$$

## Task 5 — Algorithmic Grid Approximation and Normalization

1. **Discretize.** Fix a grid $\theta_1,\dots,\theta_M$ over $(0,1]$ — e.g. `np.linspace(1e-3, 1.0, M)`,
   starting slightly above $0$ to avoid $\ln\theta=-\infty$ in the likelihood.
2. **Initialize.** Evaluate $f^{(0)}_m = \text{Beta}(8,1.5)$ density at each $\theta_m$.
3. **Sequential update.** On observing $y_k$: compute the log-normal likelihood
   $L(y_k\mid\theta_m)$ at every grid point, then form
   $\tilde f^{(k)}_m = L(y_k\mid\theta_m)\cdot f^{(k-1)}_m$.
4. **Normalize.** Use the trapezoidal rule over the bounded domain:
   $Z_k = $ `np.trapezoid(f_tilde, theta_grid)`, and set $f^{(k)}_m = \tilde f^{(k)}_m / Z_k$.
   This automatically respects the physical boundary $\theta\in(0,1]$, since the grid never
   extends outside it and the trapezoidal integral only sums contributions inside the domain
   (no probability mass leaks outside the physically valid range).
5. **Boundary handling.** Because the domain is bounded rather than $(-\infty,\infty)$, no
   special truncation of the density itself is required beyond restricting the grid to
   $(0,1]$; however near $\theta \to 0$ the log-normal likelihood can become numerically
   extreme, so working in log-space (`log f_tilde = log L + log f_prev`, subtracting the max
   before exponentiating) improves numerical stability.
6. **Estimators.** Compute $\widehat\theta^{(k)}_{\text{Bayes}} = $ `np.trapezoid(theta_grid * f_k, theta_grid)`
   and $\widehat\theta^{(k)}_{\text{MAP}} = $ `theta_grid[np.argmax(f_k)]`.
7. **Repeat** for $k=1,\dots,n$, each time using the previous normalized density as the new prior.

## Task 6 — Performance Tracking and Degradation Convergence Analysis


In [7]:
def run_shm_grid_simulation(theta_true=0.68, n_steps=15, K_nominal=50.0, sigma=0.15,
                             grid_size=2000, alpha0=8, beta0=1.5, seed=0):
    rng = np.random.default_rng(seed)
    theta_grid = np.linspace(1e-3, 1.0, grid_size)

    posterior = beta_dist.pdf(theta_grid, alpha0, beta0)
    posterior /= np.trapezoid(posterior, theta_grid)

    bayes_estimates = [np.trapezoid(theta_grid * posterior, theta_grid)]
    map_estimates = [theta_grid[np.argmax(posterior)]]
    snapshots = {0: posterior.copy()}

    for k in range(1, n_steps + 1):
        eps_k = rng.normal(0, sigma)
        y_k = theta_true * K_nominal * np.exp(eps_k)

        log_likelihood = (
            -np.log(y_k) - np.log(sigma) - 0.5 * np.log(2 * np.pi)
            - (np.log(y_k) - np.log(theta_grid) - np.log(K_nominal))**2 / (2 * sigma**2)
        )
        log_unnorm = log_likelihood + np.log(posterior + 1e-300)
        log_unnorm -= np.max(log_unnorm)
        unnorm = np.exp(log_unnorm)
        Z = np.trapezoid(unnorm, theta_grid)
        posterior = unnorm / Z

        bayes_estimates.append(np.trapezoid(theta_grid * posterior, theta_grid))
        map_estimates.append(theta_grid[np.argmax(posterior)])

        if k in {1, 2, 5, 10, 15}:
            snapshots[k] = posterior.copy()

    return theta_grid, np.array(bayes_estimates), np.array(map_estimates), snapshots

theta_grid_shm, bayes_shm, map_shm, snapshots_shm = run_shm_grid_simulation(
    theta_true=0.68, n_steps=15, K_nominal=50.0, sigma=0.15, seed=3
)

# Plot 1: evolving posterior density curves at milestones
fig6 = go.Figure()
for k_snap, dens in snapshots_shm.items():
    fig6.add_trace(go.Scatter(x=theta_grid_shm, y=dens, mode="lines", name=f"k = {k_snap}"))
fig6.add_vline(x=0.68, line_dash="dash", line_color="red", annotation_text="theta_true = 0.68")
fig6.update_layout(
    title="Evolution of the Posterior Density for Stiffness Efficiency (SHM Grid Update)",
    xaxis_title="\u03b8 (stiffness efficiency factor)",
    yaxis_title="Density",
    template="plotly_white",
    hovermode="x unified"
)
fig6.show()

# Plot 2: convergence of point estimators
steps_shm = np.arange(0, 16)
fig7 = go.Figure()
fig7.add_trace(go.Scatter(x=steps_shm, y=bayes_shm, mode="lines+markers", name="Posterior Mean (Bayes)"))
fig7.add_trace(go.Scatter(x=steps_shm, y=map_shm, mode="lines+markers", name="MAP estimate"))
fig7.add_hline(y=0.68, line_dash="dash", line_color="red", annotation_text="theta_true = 0.68")
fig7.update_layout(
    title="Convergence of Running Stiffness Estimators (n = 15 Inspections)",
    xaxis_title="Inspection step k",
    yaxis_title="Estimated stiffness efficiency",
    template="plotly_white",
    hovermode="x unified"
)
fig7.show()

print("Final posterior mean estimate:", round(bayes_shm[-1], 4))
print("Final MAP estimate:           ", round(map_shm[-1], 4))
print("True theta:                    0.68")


Final posterior mean estimate: 0.6656
Final MAP estimate:            0.6642
True theta:                    0.68


**Analysis.** At $k=0$ the optimistic $\text{Beta}(8,1.5)$ prior places its mass near
$\theta \approx 0.84$, far above the true post-impact value $\theta_{\text{true}}=0.68$.
The first one or two noisy readings (governed by $\sigma=0.15$ in log-space, a moderate
noise level) already begin pulling the density leftward, but the strong prior conviction
of health means the very first updates only partially overcome it. By roughly $k=5$
readings, the posterior mass has clearly separated from the original healthy peak and
concentrated around the true $68\%$ stiffness level; by $k=10$–$15$ the density has
narrowed substantially and both the posterior mean and MAP estimates have essentially
converged onto $\theta_{\text{true}}=0.68$, with the "optimistic prior" fully overridden
by roughly $5$–$8$ readings in a typical run. The steady narrowing of the density curves
across milestones $k\in\{0,1,2,5,10,15\}$ reflects accumulating, consistent evidence
reducing the engineers' uncertainty about remaining stiffness. For structural safety, this
matters directly: a narrow posterior concentrated well below a critical stiffness threshold
gives engineers **quantified confidence** that a real degradation event has occurred (rather
than sensor noise), supporting a timely maintenance or inspection decision rather than
either a false alarm from a single noisy reading or a dangerously slow response due to an
overly optimistic prior.


---
# Question 4

## 1. Deriving the Marginal Density

By the law of total probability, conditioning on the latent cluster label $C_i$,

$$p(x_i) = \sum_{k=1}^K P(C_i=k)\, p(x_i \mid C_i=k) = \sum_{k=1}^K \phi_k\, \mathscr N(x_i\mid \mu_k,\Sigma_k).$$

This is called a **Gaussian mixture density** because it is a convex combination (the
weights $\phi_k\ge0$ sum to $1$) of $K$ Gaussian "component" densities $\mathscr N(x_i\mid\mu_k,\Sigma_k)$.
Each component is Gaussian, but their weighted superposition need not be — the resulting
density $p(x_i)$ can be multimodal, with up to $K$ separate modes (one for each cluster's
Gaussian), which is precisely what allows the model to represent data arising from several
distinct sub-populations.

## 2. Deriving the Posterior Cluster Probability

By Bayes' rule applied to the discrete latent variable $C_i$ and the observation $X_i=x_i$,

$$P(C_i=k\mid X_i=x_i) = \frac{P(X_i=x_i\mid C_i=k)\,P(C_i=k)}{\sum_{j=1}^K P(X_i=x_i\mid C_i=j)\,P(C_i=j)}.$$

Substituting the Gaussian component model $P(X_i=x_i\mid C_i=k)=\mathscr N(x_i\mid\mu_k,\Sigma_k)$
and the cluster prior $P(C_i=k)=\phi_k$ gives

$$P(C_i=k\mid X_i=x_i) = \frac{\phi_k\,\mathscr N(x_i\mid\mu_k,\Sigma_k)}{\sum_{j=1}^K \phi_j\,\mathscr N(x_i\mid\mu_j,\Sigma_j)} \;=:\; \gamma_{ik}.$$

**Why $\gamma_{ik}$ is a posterior probability of cluster membership.** By construction it is
$P(C_i=k \mid X_i=x_i)$ — the probability, under the model, that observation $i$ belongs to
cluster $k$, updated (via Bayes' rule) after having *observed* $x_i$, starting from the prior
belief $\phi_k$ that applied before any data was seen. Just as in Questions 1–3, the prior
($\phi_k$) is revised into a posterior ($\gamma_{ik}$) using the likelihood of the observed
data given each hypothesis (here, each candidate cluster).

## 3. One-Hot Encoding of the Latent Cluster Variable

$Z_{ik}$ is a Bernoulli-type indicator: $Z_{ik}=1$ if $C_i=k$ and $Z_{ik}=0$ otherwise. For
any $\{0,1\}$-valued random variable, its conditional expectation equals the conditional
probability that it equals $1$:

$$\mathbb E[Z_{ik}\mid X_i=x_i] = 1\cdot P(Z_{ik}=1\mid X_i=x_i) + 0\cdot P(Z_{ik}=0\mid X_i=x_i) = P(C_i=k\mid X_i=x_i) = \gamma_{ik}.$$

Applying this coordinate-wise to the vector $Z_i=(Z_{i1},\dots,Z_{iK})^\top$,

$$\mathbb E[Z_i\mid X_i=x_i] = \bigl(\mathbb E[Z_{i1}\mid X_i=x_i],\dots,\mathbb E[Z_{iK}\mid X_i=x_i]\bigr)^\top = (\gamma_{i1},\dots,\gamma_{iK})^\top.$$

Hence the **soft cluster assignment vector is exactly the conditional expectation**
$\mathbb E[Z_i\mid X_i=x_i]$ — a direct probabilistic generalization of "assigning" $x_i$ to
a cluster: instead of a hard $0/1$ label, we get the expected value of the (unobserved)
one-hot label given what we observed.

## 4. From Soft Assignment to Hard Clustering

$\mathbb E[Z_i\mid X_i=x_i]=(\gamma_{i1},\dots,\gamma_{iK})$ is a *soft* assignment: it
distributes partial, fractional membership of $x_i$ across all $K$ clusters simultaneously
(each $\gamma_{ik}\in[0,1]$, summing to $1$), reflecting genuine uncertainty about which
cluster generated the point — e.g. a point near the boundary between two clusters might get
$\gamma_{i1}=0.55,\ \gamma_{i2}=0.45$. A **hard** assignment collapses this uncertainty into a
single definite label by taking the most probable cluster,
$\widehat C_i = \arg\max_k \gamma_{ik}$, discarding the information about *how confident* the
model is and about any residual ambiguity. Soft clustering retains full uncertainty
quantification (useful downstream, e.g. for weighting); hard clustering is a simpler,
interpretable summary obtained by thresholding the soft assignment at its maximum.

## 5. Conditional Expectation of the Observation Given the Cluster

Since $X_i\mid C_i=k \sim \mathscr N(\mu_k,\Sigma_k)$, and the mean of a multivariate Gaussian
$\mathscr N(\mu_k,\Sigma_k)$ is $\mu_k$ by definition of the Gaussian distribution,

$$\mathbb E[X_i\mid C_i=k] = \mu_k.$$

$\mu_k$ can therefore be interpreted as the **center of cluster $k$**: it is the expected
location of an observation *given* that it was generated by component $k$ — i.e. the point
around which cluster $k$'s Gaussian is centered.

**Comparing the two conditional expectations.** $\mathbb E[Z_i\mid X_i=x_i]$ conditions on
the *observed data point* $x_i$ and returns a distribution over *cluster labels* — it answers
"given this point, how likely is each cluster?" (soft membership of a point). $\mathbb E[X_i\mid C_i=k]$
conditions on the *cluster identity* and returns a point in $\mathbb R^d$ — it answers
"given this cluster, where do its points tend to be?" (the cluster's center). The first
moves from data $\to$ latent label (inference/clustering); the second moves from latent
label $\to$ data (generative/summary), and both directions are needed to fully specify and
use the mixture model.

## 6. The Complete-Data Likelihood

If the labels $z_i$ (equivalently, the one-hot vectors) were known, the joint probability of
data and labels factorizes over $i$ and, for each $i$, only the term corresponding to the
true cluster $k$ (where $z_{ik}=1$) contributes — which the exponent notation
$[\phi_k\mathscr N(x_i\mid\mu_k,\Sigma_k)]^{z_{ik}}$ enforces, since any term with $z_{ik}=0$
becomes a factor of $1$. Taking logarithms converts the product into a sum:

$$\ell_c = \log \prod_{i=1}^n \prod_{k=1}^K \bigl[\phi_k \mathscr N(x_i\mid\mu_k,\Sigma_k)\bigr]^{z_{ik}} = \sum_{i=1}^n \sum_{k=1}^K z_{ik}\log\bigl[\phi_k\mathscr N(x_i\mid\mu_k,\Sigma_k)\bigr] = \sum_{i=1}^n \sum_{k=1}^K z_{ik}\Bigl[\log\phi_k + \log\mathscr N(x_i\mid\mu_k,\Sigma_k)\Bigr].$$

**Why this would be easy to maximize if $z_{ik}$ were known.** With the labels fixed, the sum
decouples: for each cluster $k$, only the subset of points with $z_{ik}=1$ contributes to the
terms involving $(\phi_k,\mu_k,\Sigma_k)$. This reduces the problem to $K$ *independent*,
ordinary maximum-likelihood estimation problems, one per cluster, each solved by the standard
closed-form Gaussian MLE formulas (sample mean, sample covariance) restricted to that
cluster's points, plus a simple proportion for $\phi_k$ — no coupled or iterative
optimization is needed.

## 7. The EM Interpretation

Since $z_{ik}$ is never actually observed, the E-step replaces each unknown indicator by its
conditional expectation given the currently observed data $x_i$ and the current parameter
estimates — precisely the quantity derived in Part 3: $z_{ik}\leadsto \mathbb E[Z_{ik}\mid X_i=x_i]=\gamma_{ik}$.
Substituting $\gamma_{ik}$ for $z_{ik}$ in $\ell_c$ gives the **expected complete-data
log-likelihood**,

$$Q = \sum_{i=1}^n\sum_{k=1}^K \gamma_{ik}\Bigl[\log\phi_k + \log\mathscr N(x_i\mid\mu_k,\Sigma_k)\Bigr].$$

**Why the E-step is a conditional update of cluster membership.** Computing $\gamma_{ik}$ is
exactly the Bayesian update from Part 2 — combining the current prior $\phi_k$ with the
current Gaussian likelihood to obtain the posterior probability of membership for every
point, given the current parameter estimates. It is a "conditional update" in precisely the
same sense as the sequential updates of Questions 1–3: existing beliefs (parameters) are used
to compute an updated (posterior) belief about the latent variable ($C_i$) given the observed
data.

## 8. Parameter Updates

Maximizing $Q$ with respect to $(\phi_k,\mu_k,\Sigma_k)$ subject to $\sum_k\phi_k=1$ (via a
Lagrange multiplier for the constraint, and setting the gradients of $Q$ with respect to
$\mu_k$ and $\Sigma_k$ to zero) yields the standard **M-step** updates. Defining the
effective number of points assigned to cluster $k$,

$$N_k = \sum_{i=1}^n \gamma_{ik},$$

the maximizers are

$$\phi_k^{\text{new}} = \frac{N_k}{n}, \qquad \mu_k^{\text{new}} = \frac{1}{N_k}\sum_{i=1}^n \gamma_{ik}\,x_i, \qquad \Sigma_k^{\text{new}} = \frac{1}{N_k}\sum_{i=1}^n \gamma_{ik}\,(x_i-\mu_k^{\text{new}})(x_i-\mu_k^{\text{new}})^\top.$$

**Interpretation of $\gamma_{ik}$ as a fractional weight.** These formulas are exactly the
ordinary weighted sample mean, weighted sample covariance, and weighted proportion — as if
every point $x_i$ contributed to cluster $k$'s statistics not as a whole point, but as
$\gamma_{ik}$ "shares" of a point (and the remaining $1-\gamma_{ik}$ shares spread across the
other clusters). $N_k$ is the *effective sample size* of cluster $k$ once every point's
fractional contribution is summed.

## 9. Interpretation

Gaussian mixture clustering can be understood as a repeated cycle of conditional updating,
directly analogous to the sequential Bayesian updates in Questions 1–3. The mixture weight
$\phi_k$ plays the role of the **prior** probability that a randomly chosen point belongs to
cluster $k$, before looking at its location. The Gaussian density $\mathscr N(x_i\mid\mu_k,\Sigma_k)$
is the **likelihood**: it measures how compatible the observed location $x_i$ is with having
been generated by cluster $k$. Combining prior and likelihood via Bayes' rule produces the
responsibility $\gamma_{ik}$, the **posterior** probability of cluster $k$ given the observed
point $x_i$ — and stacking these responsibilities across all $K$ clusters gives the soft
assignment vector $\mathbb E[Z_i\mid X_i=x_i]$, the full posterior distribution over cluster
membership for that point. The **M-step** then updates the cluster parameters
$(\phi_k,\mu_k,\Sigma_k)$ by using these posterior membership probabilities as weights in
otherwise ordinary weighted-average formulas. Iterating E- and M-steps is therefore nothing
more than alternating between (i) computing posterior beliefs about latent cluster membership
given the current parameters, and (ii) updating the parameters given the current posterior
beliefs — which is exactly why **Gaussian mixture clustering is best understood as
probabilistic clustering built entirely out of conditional expectations of a latent cluster
membership variable**, rather than as an ad hoc geometric heuristic.


## 10. Computational Simulation and Out-of-Sample Validation

The class below, `GMMFinancialSegmenter`, implements the full pipeline: standardize two
continuous financial features, split into train/validation sets, fit a $K=3$ component GMM
via EM (`sklearn.mixture.GaussianMixture`), report convergence and out-of-sample
log-likelihood, and produce the three requested interactive Plotly visualizations.

**Note on the dataset.** The assignment references the Kaggle *Credit Card Dataset for
Clustering* (`arjunbhasin2013/ccdata`). This notebook's execution environment does not have
network access to Kaggle, so the class is written to accept **any** dataframe with the two
named feature columns — if you have `CC GENERAL.csv` downloaded locally (or mounted in
Colab via the Kaggle API / `kagglehub`), simply load it with `pandas.read_csv(...)` and pass
it straight into `GMMFinancialSegmenter`. To demonstrate the full pipeline end-to-end here,
we generate a synthetic two-feature dataset (`PURCHASES`, `CREDIT_LIMIT`) with three
overlapping sub-populations that mimics the multimodal structure of real credit-card
customer segments (low-spend/low-limit, moderate, and high-spend/high-limit customers).


In [8]:
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


class GMMFinancialSegmenter:
    '''
    Two-dimensional Gaussian Mixture Model segmenter for financial behavior features,
    with EM fitting via scikit-learn and interactive Plotly diagnostics.
    '''

    def __init__(self, n_components=3, feature_names=("PURCHASES", "CREDIT_LIMIT"),
                 test_size=0.2, random_state=42):
        self.n_components = n_components
        self.feature_names = list(feature_names)
        self.test_size = test_size
        self.random_state = random_state

        self.scaler = StandardScaler()
        self.gmm = GaussianMixture(
            n_components=n_components,
            covariance_type="full",
            random_state=random_state,
            n_init=5,
        )

        self.X_train_ = None
        self.X_test_ = None
        self.X_train_raw_ = None
        self.X_test_raw_ = None

    # ---------------------------------------------------------------
    def fit(self, df):
        '''Standardize features, split train/test, and fit the GMM via EM.'''
        data = df[self.feature_names].dropna().to_numpy(dtype=float)

        raw_train, raw_test = train_test_split(
            data, test_size=self.test_size, random_state=self.random_state
        )
        self.X_train_raw_, self.X_test_raw_ = raw_train, raw_test

        self.X_train_ = self.scaler.fit_transform(raw_train)
        self.X_test_ = self.scaler.transform(raw_test)

        self.gmm.fit(self.X_train_)

        print(f"Converged: {self.gmm.converged_}")
        print(f"Iterations to convergence: {self.gmm.n_iter_}")
        return self

    # ---------------------------------------------------------------
    def evaluate_out_of_sample(self):
        '''Average log-likelihood per sample on the held-out test set.'''
        avg_ll = self.gmm.score(self.X_test_)
        print(f"Average out-of-sample log-likelihood: {avg_ll:.4f}")
        return avg_ll

    # ---------------------------------------------------------------
    def plot_density_heatmap(self):
        '''Empirical 2D density heatmap of the raw training data with marginals.'''
        x_raw = self.X_train_raw_[:, 0]
        y_raw = self.X_train_raw_[:, 1]

        fig = make_subplots(
            rows=2, cols=2,
            column_widths=[0.82, 0.18], row_heights=[0.18, 0.82],
            horizontal_spacing=0.02, vertical_spacing=0.02,
            specs=[[{"type": "histogram"}, {"type": "xy"}],
                   [{"type": "histogram2d"}, {"type": "histogram"}]],
        )

        fig.add_trace(go.Histogram(x=x_raw, marker_color="steelblue", showlegend=False), row=1, col=1)
        fig.add_trace(go.Histogram2d(x=x_raw, y=y_raw, colorscale="Viridis", showscale=False),
                      row=2, col=1)
        fig.add_trace(go.Histogram(y=y_raw, marker_color="steelblue", showlegend=False), row=2, col=2)

        fig.update_xaxes(title_text=self.feature_names[0], row=2, col=1)
        fig.update_yaxes(title_text=self.feature_names[1], row=2, col=1)
        fig.update_layout(
            title="Empirical 2D Density Heatmap of Training Data (with Marginals)",
            template="plotly_white",
            bargap=0.02,
        )
        fig.show()
        return fig

    # ---------------------------------------------------------------
    def _responsibility_grid(self, x_raw, y_raw, n_grid=150, pad=0.5):
        x_min, x_max = x_raw.min() - pad * x_raw.std(), x_raw.max() + pad * x_raw.std()
        y_min, y_max = y_raw.min() - pad * y_raw.std(), y_raw.max() + pad * y_raw.std()

        xx, yy = np.meshgrid(
            np.linspace(x_min, x_max, n_grid),
            np.linspace(y_min, y_max, n_grid),
        )
        grid_raw = np.column_stack([xx.ravel(), yy.ravel()])
        grid_scaled = self.scaler.transform(grid_raw)

        responsibilities = self.gmm.predict_proba(grid_scaled)
        max_responsibility = responsibilities.max(axis=1).reshape(xx.shape)
        hard_cluster = responsibilities.argmax(axis=1).reshape(xx.shape)
        return xx, yy, max_responsibility, hard_cluster

    def _assignment_plot(self, X_raw, title):
        xx, yy, max_resp, hard_cluster = self._responsibility_grid(X_raw[:, 0], X_raw[:, 1])

        point_labels = self.gmm.predict(self.scaler.transform(X_raw))

        fig = go.Figure()
        fig.add_trace(go.Contour(
            x=xx[0], y=yy[:, 0], z=max_resp,
            colorscale="Viridis", opacity=0.75,
            contours=dict(showlines=False),
            colorbar=dict(title="max \u03b3<sub>ik</sub>"),
            name="Responsibility"
        ))
        for k in range(self.n_components):
            mask = point_labels == k
            fig.add_trace(go.Scatter(
                x=X_raw[mask, 0], y=X_raw[mask, 1],
                mode="markers", name=f"Cluster {k}",
                marker=dict(size=6, line=dict(width=0.5, color="white")),
            ))
        fig.update_layout(
            title=title,
            xaxis_title=self.feature_names[0],
            yaxis_title=self.feature_names[1],
            template="plotly_white",
        )
        fig.show()
        return fig

    def plot_training_assignment(self):
        '''Training points over a continuous contour of max posterior responsibility.'''
        return self._assignment_plot(self.X_train_raw_, "Training Data: Posterior Responsibility Contours")

    def plot_test_assignment(self):
        '''Held-out test points over the same responsibility contour boundary.'''
        return self._assignment_plot(self.X_test_raw_, "Test Data: Out-of-Sample Cluster Assignment")


def make_synthetic_financial_data(n=3000, seed=42):
    '''Synthetic stand-in for the Kaggle CC GENERAL dataset's PURCHASES vs CREDIT_LIMIT
    structure: three overlapping customer segments (low, moderate, high spenders).'''
    rng = np.random.default_rng(seed)
    n1, n2, n3 = int(0.5 * n), int(0.3 * n), n - int(0.5 * n) - int(0.3 * n)

    seg1 = rng.multivariate_normal([300, 2000], [[15000, 8000], [8000, 250000]], size=n1)
    seg2 = rng.multivariate_normal([2500, 6000], [[400000, 150000], [150000, 900000]], size=n2)
    seg3 = rng.multivariate_normal([7000, 15000], [[900000, 300000], [300000, 4000000]], size=n3)

    purchases = np.concatenate([seg1[:, 0], seg2[:, 0], seg3[:, 0]])
    credit_limit = np.concatenate([seg1[:, 1], seg2[:, 1], seg3[:, 1]])

    purchases = np.clip(purchases, 0, None)
    credit_limit = np.clip(credit_limit, 500, None)

    return pd.DataFrame({"PURCHASES": purchases, "CREDIT_LIMIT": credit_limit})


# --- Run the pipeline ---
df_financial = make_synthetic_financial_data(n=3000, seed=42)

segmenter = GMMFinancialSegmenter(n_components=3, feature_names=("PURCHASES", "CREDIT_LIMIT"))
segmenter.fit(df_financial)
segmenter.evaluate_out_of_sample()

segmenter.plot_density_heatmap()
segmenter.plot_training_assignment()
segmenter.plot_test_assignment()


Converged: True
Iterations to convergence: 3
Average out-of-sample log-likelihood: 0.2658
